In [1]:
import slangpy as spy
from pyglm import glm
import matplotlib.pyplot as plt

from bvhgs import device
from bvhgs.camera import Camera
from bvhgs.gaussian import GaussianCloud
from bvhgs.renderer import Renderer

[INFO] (rhi) layer: CreateDevice: Debug layer is enabled.
[WARN] No supported shader model found, pretending to support sm_6_0.


# Create Gaussian Buffer

In [2]:
gaussians = GaussianCloud(16)
len(gaussians)

16

# Load Module and Shader

In [3]:
module = device.load_module("renderer.slang")
module

SlangModule(
  name = renderer.slang,
  path = /Users/fangjun/Documents/stanford/bvhgs/src/bvhgs/slang/renderer.slang,
  entry_points = [
    SlangEntryPoint(name="projection", stage=compute),
    SlangEntryPoint(name="cull", stage=compute),
    SlangEntryPoint(name="rasterize", stage=compute),
  ]
)

In [4]:
program = device.link_program([module], [])
program

ShaderProgram(
  modules = [
    SlangModule(
      name = renderer.slang,
      path = /Users/fangjun/Documents/stanford/bvhgs/src/bvhgs/slang/renderer.slang,
      entry_points = [
        SlangEntryPoint(name="projection", stage=compute),
        SlangEntryPoint(name="cull", stage=compute),
        SlangEntryPoint(name="rasterize", stage=compute),
      ]
    ),
  ],
  entry_points = []
)

# Projection

## Build Slang Buffer and Camera Parameter

In [5]:
# Create a buffer for the Gaussian points.
gaussian_buf = device.create_buffer(
    element_count=len(gaussians),
    struct_type=program.reflection.g_gaussian_3d,
    usage=spy.BufferUsage.shader_resource,
)
# Store all the gaussian points in the buffer.
gaussian_cursor = spy.BufferCursor(
    program.reflection.g_gaussian_3d.type_layout.element_type_layout,
    gaussian_buf,
)
for i in range(len(gaussians)):
    gaussian_cursor[i].write(gaussians[i])
gaussian_cursor.apply()
gaussian_cursor[0].read()

{'position': {0.14119919, 0.24168971, 0.41427484},
 'rotation': {0.665669, 0.3607819, 0.7570545, 0.57758635},
 'scale': {0.46984884, 0.98957205, 0.5000082},
 'color': {0.085585326, 0.09184569, 0.33128777},
 'opacity': 0.15558163821697235,
 'sh': [{0.5535901, 0.94881195, 0.6606391},
  {0.53541625, 0.3121868, 0.55606884},
  {0.70493436, 0.8737533, 0.28209206},
  {0.26787972, 0.33551103, 0.16886085},
  {0.55732095, 0.10704167, 0.81427765},
  {0.56135786, 0.08531928, 0.070913225},
  {0.72395533, 0.122516155, 0.436081},
  {0.9310008, 0.069477916, 0.08103884},
  {0.16531481, 0.60055864, 0.83619386},
  {0.07402631, 0.8379586, 0.48928088},
  {0.47604924, 0.4055723, 0.3751082},
  {0.8224398, 0.83407, 0.65438765},
  {0.0050182203, 0.46235624, 0.08184746},
  {0.85266685, 0.7577658, 0.68755907},
  {0.635158, 0.38620168, 0.4327914}]}

In [6]:
# Create a buffer for the Gaussian2D points.
gaussian2d_buf = device.create_buffer(
    element_count=len(gaussians),
    struct_type=program.reflection.g_gaussian_2d,
    usage=spy.BufferUsage.shader_resource | spy.BufferUsage.unordered_access,
)
gaussian2d_cursor = spy.BufferCursor(
    program.reflection.g_gaussian_2d.type_layout.element_type_layout,
    gaussian2d_buf,
)
gaussian2d_cursor[0].read()

{'position': {0, 0, 0},
 'covariance': {{0, 0}, {0, 0}},
 'color': {0, 0, 0},
 'opacity': 0.0,
 'cachedInvCov': {{0, 0}, {0, 0}},
 'cachedDet': 0.0,
 'cachedNorm': 0.0}

In [7]:
camera = Camera(
    rotation=glm.quat(1, 0, 0, 0),
    translation=glm.vec3(0, 0, 0),
    sensor_size=glm.uvec2(512, 512),
    focal_length=64
)
camera.to_slang()

{'_rotation': [0.0, 0.0, 0.0, 1.0],
 '_translation': vec3( 0, 0, 0 ),
 '_sensorSize': uvec2( 512, 512 ),
 '_focalLength': 64}

## Dispatch Projection Kernel

In [8]:
ker_proj = device.create_compute_kernel(
    device.link_program([module], [module.entry_point("projection")])
)
ker_proj

ComputeKernel(0x600002704d80)

In [9]:
ker_proj.dispatch(
    thread_count=[len(gaussians), 1, 1],
    vars={
        "g_camera": camera.to_slang(),
        "g_gaussian_3d": gaussian_buf,
        "g_gaussian_2d": gaussian2d_buf
    }
)

In [10]:
gaussian2d_cursor = spy.BufferCursor(
    program.reflection.g_gaussian_2d.type_layout.element_type_layout,
    gaussian2d_buf,
)
gaussian2d_cursor[0].read()

{'position': {0.3408346, 0.5834043, 0.49997476},
 'covariance': {{5.9064603, 0.13651936}, {0.13651936, 6.0603824}},
 'color': {0.050002918, -0.79843223, 0.11642495},
 'opacity': 0.15558163821697235,
 'cachedInvCov': {{0.16939433, -0.0038158658}, {-0.0038158658, 0.16509205}},
 'cachedDet': 35.776771545410156,
 'cachedNorm': 0.02660844661295414}